# 純淨版：Raspberry Pi YOLOv8 GhostNet 優化 (無外部 Clone)

本程式完全獨立，不依賴外部 Github 專案。我們使用：
1. **官方 Ultralytics**：透過 Callback 實現稀疏化訓練。
2. **Torch-Pruning**：透過標準庫實現結構化剪枝。
3. **GhostNet**：透過 YAML 定義輕量化架構。

### 步驟 1：安裝官方環境

In [1]:
# Cell 1: 安裝官方套件
# 我們直接使用 pip 安裝，不需要 git clone 任何東西
!pip install ultralytics
!pip install torch-pruning
!pip install roboflow
!pip install onnx onnxsim

import torch
from ultralytics import YOLO
import os
import torch_pruning as tp

print(f"Torch version: {torch.__version__}")
print("環境準備完成！")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.2/70.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 117.8 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.0/21.0 MB 102.9 MB/s eta 0:00:00
  Pre

### 步驟 2：準備資料集與 GhostNet 架構

In [2]:
# Cell 2: 下載資料集
from roboflow import Roboflow

rf = Roboflow(api_key="1tVlVDPExWSPjroLUxMb")
project = rf.workspace("yolo-mahpf").project("person-eccaa-pbkcf")
version = project.version(1)
dataset = version.download("yolov8")

DATA_YAML = f"{dataset.location}/data.yaml"

# 定義實驗路徑
EXP_DIR = "/content/yolo_optimization"
os.makedirs(EXP_DIR, exist_ok=True)
print(f"Data YAML: {DATA_YAML}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to person-1 in yolov8:: 100%|██████████| 326/326 [00:00<00:00, 9343.29it/s]

Data YAML: /content/person-1/data.yaml


In [3]:
# Cell 3: 定義 GhostNet YAML
import yaml

ghost_yaml_content = {
    'nc': 80,
    'scales': {'n': [0.33, 0.25, 1024]},
    'backbone': [
        [-1, 1, 'Conv', [64, 3, 2]],
        [-1, 1, 'GhostConv', [128, 3, 2]],
        [-1, 3, 'C3Ghost', [128, True]],
        [-1, 1, 'GhostConv', [256, 3, 2]],
        [-1, 6, 'C3Ghost', [256, True]],
        [-1, 1, 'GhostConv', [512, 3, 2]],
        [-1, 6, 'C3Ghost', [512, True]],
        [-1, 1, 'GhostConv', [1024, 3, 2]],
        [-1, 3, 'C3Ghost', [1024, True]],
        [-1, 1, 'SPPF', [1024, 5]]
    ],
    'head': [
        [-1, 1, 'nn.Upsample', [None, 2, 'nearest']],
        [[-1, 6], 1, 'Concat', [1]],
        [-1, 3, 'C3Ghost', [512]],
        [-1, 1, 'nn.Upsample', [None, 2, 'nearest']],
        [[-1, 4], 1, 'Concat', [1]],
        [-1, 3, 'C3Ghost', [256]],
        [-1, 1, 'GhostConv', [256, 3, 2]],
        [[-1, 12], 1, 'Concat', [1]],
        [-1, 3, 'C3Ghost', [512]],
        [-1, 1, 'GhostConv', [512, 3, 2]],
        [[-1, 9], 1, 'Concat', [1]],
        [-1, 3, 'C3Ghost', [1024]],
        [[15, 18, 21], 1, 'Detect', ['nc']]
    ]
}

GHOST_YAML_PATH = os.path.join(EXP_DIR, "yolov8n-ghost.yaml")
with open(GHOST_YAML_PATH, 'w') as f:
    yaml.dump(ghost_yaml_content, f, sort_keys=False)

print(f"GhostNet Config Created: {GHOST_YAML_PATH}")

GhostNet Config Created: /content/yolo_optimization/yolov8n-ghost.yaml


### 步驟 3：訓練 Baseline
我們先用官方功能訓練一個標準的 GhostNet。

In [4]:
# Cell 4: 訓練 Baseline
from ultralytics import YOLO

BASELINE_NAME = "ghost_baseline_320"

# 1. 建立模型
# 先載入 GhostNet 架構 (隨機權重)
model = YOLO(GHOST_YAML_PATH)

# 2. 載入預訓練權重 (Transfer Learning)
# 這樣做比 model.load() 更穩健，它會自動適配不同層
model = YOLO(GHOST_YAML_PATH).load("yolov8n.pt")

print("🚀 開始訓練 Baseline...")
model.train(
    data=DATA_YAML,
    imgsz=320,
    epochs=150,
    batch=16,
    project=EXP_DIR,     # 確保這裡是指向 /content/yolo_optimization
    name=BASELINE_NAME,
    pretrained=True,      # 讓 YOLO 知道我們要用預訓練權重
    exist_ok=True
)

BASELINE_PT = f"{EXP_DIR}/{BASELINE_NAME}/weights/best.pt"
print(f"✅ Baseline Training Complete: {BASELINE_PT}")

Transferred 119/559 items from pretrained weights
🚀 開始訓練 Baseline...
Ultralytics 8.3.248 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/person-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/yolo_optimization/yolov8n-ghost.yaml, momentum=0.937, mosaic=1.0, multi_scale=False, name=ghost_baseline_320, nbs=64, nms=Fal

In [5]:
# ================================
# ✅ Block: Train Non-Ghost Baseline (YOLOv8n original conv)
# ================================
from ultralytics import YOLO
import os

NON_GHOST_NAME = "yolov8n_baseline_320"
IMGSZ = 320
EPOCHS = 20
BATCH = 16

print("🚀 Training Non-Ghost Baseline (YOLOv8n original)...")

model_ng = YOLO("yolov8n.pt")  # ✅ 原生 YOLOv8n（非 GhostNet）

model_ng.train(
    data=DATA_YAML,
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=BATCH,
    project=EXP_DIR,
    name=NON_GHOST_NAME,
    pretrained=True,
    exist_ok=True
)

NON_GHOST_PT = f"{EXP_DIR}/{NON_GHOST_NAME}/weights/best.pt"
print(f"✅ Non-Ghost Baseline Training Complete: {NON_GHOST_PT}")


🚀 Training Non-Ghost Baseline (YOLOv8n original)...
Ultralytics 8.3.248 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/person-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8n_baseline_320, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, ove

### 步驟 4：定義稀疏化 Callback (取代 train.py)
這就是魔法所在。我們定義一個函數，在每次 Backward 之後被呼叫，手動加上 L1 正則化。

In [6]:
# Cell 5 (修正版): 定義稀疏化 Callback (改用環境變數)
import os
import torch

def sparsity_callback(trainer):
    # 1. 從環境變數讀取 L1 強度 (預設為 0，代表不啟用)
    # 我們繞過 trainer.args，直接讀取系統設定，避開語法檢查
    l1_lambda = float(os.getenv('YOLO_SPARSITY', 0))

    if l1_lambda > 0:
        # 遍歷模型所有層
        for k, m in trainer.model.named_modules():
            # 只針對 Batch Normalization 層
            if isinstance(m, torch.nn.BatchNorm2d):
                # 確保有梯度
                if hasattr(m.weight, 'grad') and m.weight.grad is not None:
                    # 核心公式：Gradient += Lambda * sign(Weight)
                    m.weight.grad.data.add_(l1_lambda * torch.sign(m.weight.data))

print("✅ 稀疏化 Callback 已定義 (將讀取 YOLO_SPARSITY 環境變數)。")

✅ 稀疏化 Callback 已定義 (將讀取 YOLO_SPARSITY 環境變數)。


### 步驟 5：執行稀疏化訓練 (Sparse Training)

In [7]:
# Cell 6 (修正版): 執行稀疏化訓練
SPARSE_NAME = "ghost_sparse_run"

print(f"Loading Baseline: {BASELINE_PT}")
sparse_model = YOLO(BASELINE_PT)

# 1. 註冊 Callback
sparse_model.add_callback("on_train_batch_end", sparsity_callback)

# 2. 設定環境變數 (開啟稀疏化)
os.environ['YOLO_SPARSITY'] = '0.0001'
print(f"🔥 稀疏化模式已啟動: YOLO_SPARSITY = {os.environ['YOLO_SPARSITY']}")

# 3. 啟動訓練
# 注意：這裡【不要】再傳入 sparsity=...，否則會報錯
sparse_model.train(
    data=DATA_YAML,
    imgsz=320,
    epochs=50,
    batch=16,
    lr0=0.001,
    amp=False,  # [重要] 必須關閉混合精度，否則 L1 無效
    project=EXP_DIR,
    name=SPARSE_NAME,
    exist_ok=True
)

# 4. 訓練結束後關閉稀疏化 (歸零)
os.environ['YOLO_SPARSITY'] = '0'

SPARSE_PT = f"{EXP_DIR}/{SPARSE_NAME}/weights/best.pt"
print(f"✅ Sparse Training Complete: {SPARSE_PT}")

Loading Baseline: /content/yolo_optimization/ghost_baseline_320/weights/best.pt
🔥 稀疏化模式已啟動: YOLO_SPARSITY = 0.0001
Ultralytics 8.3.248 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=False, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/person-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/yolo_optimization/ghost_baseline_320/weights/best.pt, momentum=0.937, mosaic=1

In [8]:
# ================================
# ✅ Export Sparsified model (after sparse training) to NCNN folder
# 這格要放在「稀疏化訓練完成」後面
# ================================

import os, glob, shutil, zipfile
from ultralytics import YOLO

IMG_SIZE = 320
EXP_DIR = "/content/yolo_optimization"
OUT_NAME = "sparse_ncnn_folder"   # 你要給學姊的稀疏化版本

# 1) 直接指定：稀疏化訓練後的 best.pt
#    你稀疏化那格跑完請存 SPARSE_PT（例如：SPARSE_PT = ".../runs/detect/sparse/weights/best.pt"）
if "SPARSE_PT" in globals() and isinstance(SPARSE_PT, str) and os.path.exists(SPARSE_PT):
    pt_path = SPARSE_PT
else:
    # 如果你還沒存 SPARSE_PT，我也給你一個 fallback：找含 sparse 關鍵字的 best.pt
    candidates = glob.glob("/content/yolo_optimization/ghost_sparse_run/weights/best.pt", recursive=True)
    candidates_sparse = [p for p in candidates if ("sparse" in p.lower() or "spars" in p.lower())]
    if not candidates_sparse:
        raise FileNotFoundError(
            "找不到 SPARSE_PT，也找不到路徑含 sparse 的 best.pt。\n"
            "請在稀疏化訓練那格最後加上：SPARSE_PT = <稀疏化 best.pt 的路徑>"
        )
    pt_path = max(candidates_sparse, key=os.path.getctime)

print("✅ Sparsified PT:", pt_path)

# 2) export(format="ncnn") 產生學姊要的資料夾格式
model = YOLO(pt_path)
ncnn_dir = model.export(format="ncnn", imgsz=IMG_SIZE, half=True)
print("✅ NCNN folder:", ncnn_dir)

# 3) 整理輸出到固定資料夾
out_dir = os.path.join(EXP_DIR, OUT_NAME)
if os.path.exists(out_dir):
    shutil.rmtree(out_dir)
shutil.copytree(ncnn_dir, out_dir)

# 4) 打包 zip 給學姊
zip_path = os.path.join(EXP_DIR, f"{OUT_NAME}.zip")
if os.path.exists(zip_path):
    os.remove(zip_path)

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(out_dir):
        for fn in files:
            full = os.path.join(root, fn)
            rel = os.path.relpath(full, EXP_DIR)
            z.write(full, arcname=rel)

print("📁 Saved to:", out_dir)
print("📦 ZIP:", zip_path)


✅ Sparsified PT: /content/yolo_optimization/ghost_sparse_run/weights/best.pt
Ultralytics 8.3.248 🚀 Python-3.12.12 torch-2.9.0+cu126 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLOv8n-ghost summary (fused): 146 layers, 1,714,466 parameters, 0 gradients, 5.0 GFLOPs

PyTorch: starting from '/content/yolo_optimization/ghost_sparse_run/weights/best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 6, 2100) (3.5 MB)
requirements: Ultralytics requirement ['ncnn'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 1 package in 353ms
Prepared 1 package in 105ms
Installed 1 package in 3ms
 + ncnn==1.0.20250916

requirements: AutoUpdate success ✅ 0.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

requirements: Ultralytics requirement ['pnnx'] not found, attempting AutoUpdate...

In [9]:
import torch
from ultralytics import YOLO

SPARSE_PT = "/content/yolo_optimization/ghost_sparse_run/weights/best.pt"
IMGSZ = 320

yolo = YOLO(SPARSE_PT)
model = yolo.model.to("cpu").eval()
x = torch.randn(1, 3, IMGSZ, IMGSZ)

with torch.no_grad():
    out = model(x)

print("out type:", type(out))
if isinstance(out, (list, tuple)):
    print("len(out):", len(out))
    print("out[0] type:", type(out[0]))
    print("out[0] shape:", getattr(out[0], "shape", None))
elif isinstance(out, dict):
    print("keys:", out.keys())
    first_tensor = None
    for v in out.values():
        if torch.is_tensor(v):
            first_tensor = v
            break
    print("first tensor shape:", getattr(first_tensor, "shape", None))
else:
    print("shape:", getattr(out, "shape", None))


out type: <class 'tuple'>
len(out): 2
out[0] type: <class 'torch.Tensor'>
out[0] shape: torch.Size([1, 6, 2100])


In [10]:
import torch
import torch_pruning as tp
from ultralytics import YOLO
import inspect
from collections import Counter

SPARSE_PT = "/content/yolo_optimization/ghost_sparse_run/weights/best.pt"
IMGSZ = 320

yolo = YOLO(SPARSE_PT)
model = yolo.model.to("cpu").eval()
x = torch.randn(1, 3, IMGSZ, IMGSZ)

def forward_only_pred(m, *inputs):
    # 兼容兩種情況：
    # 1) inputs == (x,)  -> OK
    # 2) inputs == ((x,),) 或 inputs == ([x],) -> 需要再解包一次
    if len(inputs) == 1 and isinstance(inputs[0], (tuple, list)) and len(inputs[0]) == 1 and torch.is_tensor(inputs[0][0]):
        inputs = (inputs[0][0],)

    y = m(*inputs)
    # model(x) 會回傳 (pred, aux) 之類的 tuple，取 pred tensor
    if isinstance(y, (tuple, list)):
        y = y[0]
    return y

DG = tp.DependencyGraph()
sig = inspect.signature(DG.build_dependency)

kwargs = {"model": model, "example_inputs": (x,)}

# 依照你的 torch-pruning 版本支援的參數名掛上 forward_fn / output_transform
if "forward_fn" in sig.parameters:
    kwargs["forward_fn"] = forward_only_pred
elif "forward" in sig.parameters:
    kwargs["forward"] = forward_only_pred
elif "output_transform" in sig.parameters:
    kwargs["output_transform"] = lambda out: out[0] if isinstance(out, (tuple, list)) else out

DG.build_dependency(**kwargs)

mods = list(DG.module2node.keys())
print("DG modules:", len(mods))

type_counts = Counter(type(m).__name__ for m in mods)
print("Top module types in DG:", type_counts.most_common(15))

conv2d_cnt = sum(isinstance(m, torch.nn.Conv2d) for m in mods)
bn_cnt = sum(isinstance(m, torch.nn.BatchNorm2d) for m in mods)
print("Conv2d in DG:", conv2d_cnt)
print("BatchNorm2d in DG:", bn_cnt)


DG modules: 1
Top module types in DG: [('Conv2d', 1)]
Conv2d in DG: 1
BatchNorm2d in DG: 0


### 步驟 6：執行剪枝 (取代 prune.py)
使用 `torch_pruning` 函式庫，它比原本的腳本更穩定、更現代。

In [26]:
import os
import yaml
from ultralytics import YOLO

EXP_DIR = "/content/yolo_optimization"
DATA_YAML = "/content/person-1/data.yaml"  # 你的資料集 yaml
SRC_PT = "/content/yolo_optimization/ghost_baseline_320/weights/best.pt"

IMGSZ = 320
WIDTH_SCALE = 0.7     # 等效 prune30
EPOCHS = 150
BATCH = 16
NAME = "ghost_prune30_widthscale_320"

os.makedirs(EXP_DIR, exist_ok=True)

# 1) 讀原模型的 yaml config
src = YOLO(SRC_PT)
cfg = src.model.yaml.copy()

# 2) 寬度縮放：優先處理 yolov8 常見的 cfg 格式
if "width_multiple" in cfg:
    cfg["width_multiple"] = float(cfg["width_multiple"]) * WIDTH_SCALE
elif "scales" in cfg and isinstance(cfg["scales"], dict):
    # scales 形式：[depth, width, max_channels]，把 width 乘上 0.7
    for k, v in cfg["scales"].items():
        if isinstance(v, (list, tuple)) and len(v) >= 2:
            v = list(v)
            v[1] = float(v[1]) * WIDTH_SCALE
            cfg["scales"][k] = v
else:
    cfg["width_multiple"] = WIDTH_SCALE

OUT_YAML = f"{EXP_DIR}/ghost_prune30.yaml"
with open(OUT_YAML, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("✅ Saved prune30 yaml:", OUT_YAML)

# 3) 用新 yaml 建模並訓練（結構真的變小）
m = YOLO(OUT_YAML)
m.train(
    data=DATA_YAML,
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=BATCH,
    project=EXP_DIR,
    name=NAME,
    pretrained=False,
    exist_ok=True
)

PRUNE30_PT = f"{EXP_DIR}/{NAME}/weights/best.pt"
print("✅ Prune30 best.pt:", PRUNE30_PT)


✅ Saved prune30 yaml: /content/yolo_optimization/ghost_prune30.yaml
WARNING ⚠️ no model scale passed. Assuming scale='n'.
Ultralytics 8.3.248 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/person-1/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/yolo_optimization/ghost_prune30.yaml, momentum=0.937, mosaic=1.0, multi_scal

### 步驟 7：微調與導出 ONNX

In [30]:
import os
from ultralytics import YOLO

FINETUNE_NAME = "ghost_width07_finetune"
IMGSZ = 320

# 1) 用 width×0.7 的 best.pt 當起點微調
model = YOLO(PRUNE30_PT)
train_res = model.train(
    data=DATA_YAML,
    imgsz=IMGSZ,
    epochs=300,
    lr0=0.01,
    project=EXP_DIR,
    name=FINETUNE_NAME,
    resume=False,
    exist_ok=True
)

# 2) 取得 best.pt 路徑（ultralytics 會回傳 save_dir）
save_dir = str(train_res.save_dir)  # e.g. /content/yolo_optimization/ghost_width07_finetune
best_pt = os.path.join(save_dir, "weights", "best.pt")
print("✅ best.pt:", best_pt)

# 3) 匯出 ONNX
best_model = YOLO(best_pt)
onnx_path = best_model.export(format="onnx", imgsz=IMGSZ, opset=12, simplify=True)
print("🎉 ONNX exported:", onnx_path)


Ultralytics 8.3.248 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/person-1/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/yolo_optimization/ghost_prune30_widthscale_320/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=ghost_width07_finetune, nbs=64, nms=False, opset=None, optimize=False, optimizer=a

In [13]:
# Cell 11 (終極解決版): 下載官方預編譯工具 (跳過編譯錯誤)
import os
import shutil
import urllib.request
import zipfile

# 1. 設定工具目錄
NCNN_ROOT = "/content/ncnn_binaries"
# 清理舊目錄以防萬一
if os.path.exists(NCNN_ROOT): shutil.rmtree(NCNN_ROOT)
os.makedirs(NCNN_ROOT, exist_ok=True)

# 2. 下載官方 Release (適用於 Colab 的 Ubuntu 環境)
# 使用 20240410 版本
url = "https://github.com/Tencent/ncnn/releases/download/20240410/ncnn-20240410-ubuntu-2204.zip"
zip_path = os.path.join(NCNN_ROOT, "ncnn.zip")

print("⬇️ 正在下載官方 NCNN 工具 (由 Tencent 提供)...")

try:
    urllib.request.urlretrieve(url, zip_path)
    print("✅ 下載完成，正在解壓縮...")

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(NCNN_ROOT)

    # 3. 定義工具路徑
    # 解壓後會有一層資料夾 ncnn-20240410-ubuntu-2204
    base_dir = os.path.join(NCNN_ROOT, "ncnn-20240410-ubuntu-2204", "bin")

    ONNX2NCNN = os.path.join(base_dir, "onnx2ncnn")
    NCNNOPTIMIZE = os.path.join(base_dir, "ncnnoptimize")

    # 4. 驗證與授權
    if os.path.exists(ONNX2NCNN):
        print(f"✅ 工具就緒: {ONNX2NCNN}")
        !chmod +x {ONNX2NCNN} {NCNNOPTIMIZE}
        print("🎉 環境準備完成！請執行下一個 Cell 進行轉換。")
    else:
        print("❌ 錯誤：解壓縮後找不到 onnx2ncnn，請檢查下載連結。")

except Exception as e:
    print(f"❌ 下載或解壓縮失敗: {e}")

⬇️ 正在下載官方 NCNN 工具 (由 Tencent 提供)...
✅ 下載完成，正在解壓縮...
✅ 工具就緒: /content/ncnn_binaries/ncnn-20240410-ubuntu-2204/bin/onnx2ncnn
🎉 環境準備完成！請執行下一個 Cell 進行轉換。


In [14]:
# Cell 12: 執行 ONNX -> NCNN (FP16 優化版)
import glob
import os

# ================= 1. 自動尋找 ONNX 檔案 =================
if 'ONNX_MODEL_PATH' in locals() and os.path.exists(ONNX_MODEL_PATH):
    TARGET_ONNX = ONNX_MODEL_PATH
else:
    # 搜尋所有可能的 best.onnx
    search_path = "/content/yolo_optimization/**/best.onnx"
    candidates = glob.glob(search_path, recursive=True)
    if candidates:
        TARGET_ONNX = max(candidates, key=os.path.getctime)
    else:
        # 如果真的找不到，試著去 drive 找
        drive_search = "/content/drive/MyDrive/**/best.onnx"
        candidates = glob.glob(drive_search, recursive=True)
        if candidates:
            TARGET_ONNX = max(candidates, key=os.path.getctime)
        else:
            raise FileNotFoundError("❌ 找不到 best.onnx，請確認 Cell 9 是否執行成功。")

print(f"🚀 目標模型: {TARGET_ONNX}")
MODEL_DIR = os.path.dirname(TARGET_ONNX)
BASENAME = os.path.splitext(os.path.basename(TARGET_ONNX))[0]

# 定義輸出檔名
NCNN_PARAM = os.path.join(MODEL_DIR, f"{BASENAME}.param")
NCNN_BIN = os.path.join(MODEL_DIR, f"{BASENAME}.bin")
NCNN_OPT_PARAM = os.path.join(MODEL_DIR, f"{BASENAME}_fp16.param")
NCNN_OPT_BIN = os.path.join(MODEL_DIR, f"{BASENAME}_fp16.bin")

# ================= 2. 轉換 (ONNX -> NCNN) =================
print("\n[1/2] 轉換 ONNX -> NCNN...")
!{ONNX2NCNN} "{TARGET_ONNX}" "{NCNN_PARAM}" "{NCNN_BIN}"

# ================= 3. 優化 (FP16) =================
print("\n[2/2] 優化模型結構 (FP16 Mode)...")
# 參數說明:
# 0 = FP32 (標準)
# 65536 = FP16 (半精度，更小更快)
!{NCNNOPTIMIZE} "{NCNN_PARAM}" "{NCNN_BIN}" "{NCNN_OPT_PARAM}" "{NCNN_OPT_BIN}" 65536

print("-" * 60)
if os.path.exists(NCNN_OPT_BIN):
    print("🎉 轉換成功！請下載以下兩個檔案到 Raspberry Pi：")
    print(f"1. {NCNN_OPT_PARAM} (結構檔)")
    print(f"2. {NCNN_OPT_BIN} (權重檔)")
    print("\n💡 提示：這是一組 FP16 優化模型，速度非常快，且無需量化校準。")
else:
    print("❌ 轉換失敗，請檢查上方錯誤訊息。")
print("-" * 60)

🚀 目標模型: /content/yolo_optimization/ghost_width07_finetune/weights/best.onnx

[1/2] 轉換 ONNX -> NCNN...

[2/2] 優化模型結構 (FP16 Mode)...
Input layer images without shape info, shape_inference skipped
Input layer images without shape info, estimate_memory_footprint skipped
------------------------------------------------------------
🎉 轉換成功！請下載以下兩個檔案到 Raspberry Pi：
1. /content/yolo_optimization/ghost_width07_finetune/weights/best_fp16.param (結構檔)
2. /content/yolo_optimization/ghost_width07_finetune/weights/best_fp16.bin (權重檔)

💡 提示：這是一組 FP16 優化模型，速度非常快，且無需量化校準。
------------------------------------------------------------


In [31]:
import cv2
import time
import numpy as np
import pandas as pd
import os
from pathlib import Path
from ultralytics import YOLO

# --- 修改點 1: 強制隱藏 GPU，確保環境只使用 CPU ---
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
print("🖥️ 強制設定為 CPU 模式 (CUDA_VISIBLE_DEVICES=-1)")
# ------------------------------------------------

VIDEO_PATH = "/content/786289740.079699.mp4"
DATA_YAML = "/content/person-1/data.yaml"

models_list = [
    {"name": "0) YOLOv8n Baseline (non-ghost, PT)", "type": "pt", "path": "/content/yolo_optimization/yolov8n_baseline_320/weights/best.pt", "imgsz": 320},
    {"name": "1) Ghost Baseline (PT)", "type": "pt", "path": "/content/yolo_optimization/ghost_baseline_320/weights/best.pt", "imgsz": 320},
    {"name": "2) Ghost Sparse (PT)", "type": "pt", "path": "/content/yolo_optimization/ghost_sparse_run/weights/best.pt", "imgsz": 320},
    {"name": "3) Ghost width×0.7 + Finetune (PT)", "type": "pt", "path": "/content/yolo_optimization/ghost_width07_finetune/weights/best.pt", "imgsz": 320},
    {"name": "4) Sparse (NCNN)", "type": "ncnn_size_only", "ncnn_dir": "/content/yolo_optimization/sparse_ncnn_folder"},
    {"name": "5) Pruned (NCNN)", "type": "ncnn_size_only", "ncnn_dir": "/content/yolo_optimization/optimized_ncnn_folder"},
]


def get_file_size_mb(path: str) -> float:
    return os.path.getsize(path) / (1024 * 1024) if os.path.exists(path) else 0.0

def find_ncnn_files(ncnn_dir: str):
    p = Path(ncnn_dir)
    param1 = p / "model.ncnn.param"
    bin1 = p / "model.ncnn.bin"
    if param1.exists() and bin1.exists():
        return str(param1), str(bin1)
    param2 = p / "best.param"
    bin2 = p / "best.bin"
    if param2.exists() and bin2.exists():
        return str(param2), str(bin2)
    params = list(p.glob("*.param"))
    bins = list(p.glob("*.bin"))
    if params and bins:
        return str(params[0]), str(bins[0])
    return None, None

def get_ncnn_total_size_mb(ncnn_dir: str) -> float:
    param_path, bin_path = find_ncnn_files(ncnn_dir)
    if not param_path or not bin_path:
        return 0.0
    return get_file_size_mb(param_path) + get_file_size_mb(bin_path)

def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    if inter <= 0:
        return 0.0
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - inter
    return float(inter / union) if union > 0 else 0.0

class SimpleTracker:
    def __init__(self, iou_th=0.3):
        self.iou_th = iou_th
        self.tracks = {}
        self.next_id = 1

    def update(self, detections):
        new_tracks = {}
        used = set()
        for det in detections:
            best_iou = 0.0
            best_id = None
            for tid, box in self.tracks.items():
                if tid in used:
                    continue
                iou = compute_iou(det, box)
                if iou > best_iou:
                    best_iou = iou
                    best_id = tid
            if best_id is not None and best_iou >= self.iou_th:
                new_tracks[best_id] = det
                used.add(best_id)
            else:
                new_tracks[self.next_id] = det
                self.next_id += 1
        self.tracks = new_tracks
        return new_tracks

def run_pt_fps(pt_path: str, video_path: str, imgsz: int, max_frames=150) -> float:
    model = YOLO(pt_path)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError("Video open failed")

    # Warmup
    for _ in range(5):
        ret, frame = cap.read()
        if ret:
            # --- 修改點 2: 加入 device='cpu' ---
            _ = model(frame, imgsz=imgsz, verbose=False, device='cpu')
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

    tracker = SimpleTracker()
    n = 0
    t0 = time.time()
    while True:
        ret, frame = cap.read()
        if not ret or n >= max_frames:
            break
        # --- 修改點 3: 加入 device='cpu' ---
        results = model(frame, imgsz=imgsz, verbose=False, device='cpu')
        dets = []
        for r in results:
            for b in r.boxes:
                if int(b.cls[0]) == 0:
                    dets.append(b.xyxy[0].tolist())
        tracker.update(dets)
        n += 1
    cap.release()
    dt = time.time() - t0
    return (n / dt) if dt > 0 else 0.0

def run_pt_metrics(pt_path: str, data_yaml: str, imgsz: int):
    model = YOLO(pt_path)
    # --- 修改點 4: 加入 device='cpu' ---
    metrics = model.val(data=data_yaml, imgsz=imgsz, verbose=False, plots=False, device='cpu')
    return {
        "mAP@50": float(metrics.box.map50),
        "mAP@50-95": float(metrics.box.map),
        "Precision": float(metrics.box.mp),
        "Recall": float(metrics.box.mr),
    }

rows = []
print("🎬 Benchmark (PT: FPS + mAP/Precision/Recall) + (NCNN: Size only)")
print("VIDEO:", VIDEO_PATH)
print("DATA :", DATA_YAML)

for m in models_list:
    name = m["name"]
    mtype = m["type"]

    if mtype == "pt":
        pt_path = m["path"]
        imgsz = m["imgsz"]
        if not os.path.exists(pt_path):
            rows.append({
                "Stage": name, "Type": "pt", "imgsz": imgsz,
                "Size (MB)": 0.0, "FPS (Colab)": "N/A",
                "mAP@50": "N/A", "mAP@50-95": "N/A", "Precision": "N/A", "Recall": "N/A",
                "Status": "PT Not Found"
            })
            continue

        size_mb = get_file_size_mb(pt_path)

        try:
            fps = run_pt_fps(pt_path, VIDEO_PATH, imgsz=imgsz, max_frames=150)
        except Exception as e:
            fps = None
            fps_err = str(e)[:60]
        else:
            fps_err = ""

        try:
            met = run_pt_metrics(pt_path, DATA_YAML, imgsz=imgsz)
        except Exception as e:
            met = None
            met_err = str(e)[:60]
        else:
            met_err = ""

        status = "Success"
        if fps is None and met is None:
            status = f"Error: fps({fps_err}) val({met_err})"
        elif fps is None:
            status = f"Val OK / FPS Error: {fps_err}"
        elif met is None:
            status = f"FPS OK / Val Error: {met_err}"

        rows.append({
            "Stage": name,
            "Type": "pt",
            "imgsz": imgsz,
            "Size (MB)": round(size_mb, 2),
            "FPS (Colab)": round(fps, 2) if fps is not None else "N/A",
            "mAP@50": round(met["mAP@50"], 4) if met else "N/A",
            "mAP@50-95": round(met["mAP@50-95"], 4) if met else "N/A",
            "Precision": round(met["Precision"], 4) if met else "N/A",
            "Recall": round(met["Recall"], 4) if met else "N/A",
            "Status": status
        })

    elif mtype == "ncnn_size_only":
        ncnn_dir = m["ncnn_dir"]
        size_mb = get_ncnn_total_size_mb(ncnn_dir) if os.path.exists(ncnn_dir) else 0.0
        rows.append({
            "Stage": name,
            "Type": "ncnn",
            "imgsz": "N/A",
            "Size (MB)": round(size_mb, 2),
            "FPS (Colab)": "N/A",
            "mAP@50": "-",
            "mAP@50-95": "-",
            "Precision": "-",
            "Recall": "-",
            "Status": "Size Only (FPS/mAP on Pi)"
        })

df = pd.DataFrame(rows)
print("\n" + "="*110)
print("🏁 Summary Table (CPU Benchmark)")
print("="*110)
print(df.to_markdown(index=False))

OUT_CSV = "/content/yolo_optimization/colab_benchmark_summary_cpu.csv"
df.to_csv(OUT_CSV, index=False)
print("\n✅ Saved CSV:", OUT_CSV)

BENCHMARK_TABLE = df

🖥️ 強制設定為 CPU 模式 (CUDA_VISIBLE_DEVICES=-1)
🎬 Benchmark (PT: FPS + mAP/Precision/Recall) + (NCNN: Size only)
VIDEO: /content/786289740.079699.mp4
DATA : /content/person-1/data.yaml
Ultralytics 8.3.248 🚀 Python-3.12.12 torch-2.9.0+cu126 CPU (Intel Xeon CPU @ 2.00GHz)
Model summary (fused): 72 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1643.3±721.5 MB/s, size: 75.8 KB)
val: Scanning /content/person-1/valid/labels.cache... 32 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 32/32 56.6Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.3it/s 1.6s
                   all         32         71       0.83      0.638       0.74      0.449
Speed: 0.9ms preprocess, 41.6ms inference, 0.0ms loss, 2.7ms postprocess per image
Ultralytics 8.3.248 🚀 Python-3.12.12 torch-2.9.0+cu126 CPU (Intel Xeon CPU @ 2.00GHz)
YOLOv8n-ghost summary (fused): 146 layers, 1,714,466 param

In [32]:
import os
import shutil
import zipfile
from pathlib import Path
from datetime import datetime

paths = [
    "/content/yolo_optimization/ghost_baseline_320/weights/best.pt",
    "/content/yolo_optimization/ghost_sparse_run/weights/best.pt",
    "/content/yolo_optimization/ghost_width07_finetune/weights/best.pt",
]

out_root = Path("/content/yolo_optimization/export_pt_bundle")
if out_root.exists():
    shutil.rmtree(out_root)
out_root.mkdir(parents=True, exist_ok=True)

name_map = {
    "ghost_baseline_320": "baseline_best.pt",
    "ghost_sparse_run": "sparse_best.pt",
    "ghost_width07_finetune": "pruned_finetune_best.pt",
}

copied = []
for p in paths:
    src = Path(p)
    if not src.exists():
        print("Missing:", p)
        continue
    tag = None
    for key in name_map:
        if key in p:
            tag = key
            break
    new_name = name_map.get(tag, f"model_{len(copied)+1}_best.pt")
    dst = out_root / new_name
    shutil.copy2(src, dst)
    copied.append(dst)
    print("Copied:", src, "->", dst)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_path = Path("/content/yolo_optimization") / f"pt_bundle_{ts}.zip"
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for f in copied:
        z.write(f, arcname=f.name)

print("ZIP:", zip_path)


Copied: /content/yolo_optimization/ghost_baseline_320/weights/best.pt -> /content/yolo_optimization/export_pt_bundle/baseline_best.pt
Copied: /content/yolo_optimization/ghost_sparse_run/weights/best.pt -> /content/yolo_optimization/export_pt_bundle/sparse_best.pt
Copied: /content/yolo_optimization/ghost_width07_finetune/weights/best.pt -> /content/yolo_optimization/export_pt_bundle/pruned_finetune_best.pt
ZIP: /content/yolo_optimization/pt_bundle_20260105_150238.zip


In [33]:
import torch
from ultralytics import YOLO

BASELINE_PT = "/content/yolo_optimization/yolov8n_baseline_320/weights/best.pt"
GHOST_PT    = "/content/yolo_optimization/ghost_baseline_320/weights/best.pt"
SPARSE_PT   = "/content/yolo_optimization/export_pt_bundle/sparse_best.pt"
PRUNED_PT   = "/content/yolo_optimization/ghost_prune30_widthscale_320/weights/best.pt"

IMGSZ = 320

def count_params(m: torch.nn.Module) -> int:
    return sum(p.numel() for p in m.parameters())

def profile_flops_gflops(m: torch.nn.Module, imgsz: int) -> float:
    try:
        from thop import profile
    except Exception:
        raise RuntimeError("thop 沒安裝，先跑：!pip -q install thop")
    m = m.eval()
    x = torch.zeros(1, 3, imgsz, imgsz)
    with torch.no_grad():
        flops, params = profile(m, inputs=(x,), verbose=False)
    return flops / 1e9

def pct_reduction(base, new):
    return (base - new) / base * 100 if base > 0 else 0.0

models = [
    ("Non-Ghost Baseline", BASELINE_PT),
    ("Ghost Baseline", GHOST_PT),
    ("Ghost Sparse", SPARSE_PT),
    ("Ghost Pruned+Finetune", PRUNED_PT),
]

rows = []

for name, path in models:
    m = YOLO(path)
    params = count_params(m.model)
    gflops = profile_flops_gflops(m.model, IMGSZ)
    rows.append((name, path, params, gflops))

print(f"=== imgsz={IMGSZ} ===\n")
for name, path, params, gflops in rows:
    print(f"{name}:")
    print(f"  path  : {path}")
    print(f"  params: {params:,}")
    print(f"  flops : {gflops:.2f} GFLOPs\n")

base_params = rows[0][2]
base_flops  = rows[0][3]

print("=== Reduction vs Non-Ghost Baseline ===\n")
for name, path, params, gflops in rows[1:]:
    print(f"{name}:")
    print(f"  Params reduction: {base_params - params:,} ({pct_reduction(base_params, params):.2f}%)")
    print(f"  FLOPs  reduction: {base_flops - gflops:.2f} ({pct_reduction(base_flops, gflops):.2f}%)\n")


=== imgsz=320 ===

Non-Ghost Baseline:
  path  : /content/yolo_optimization/yolov8n_baseline_320/weights/best.pt
  params: 3,011,238
  flops : 1.02 GFLOPs

Ghost Baseline:
  path  : /content/yolo_optimization/ghost_baseline_320/weights/best.pt
  params: 1,719,354
  flops : 0.64 GFLOPs

Ghost Sparse:
  path  : /content/yolo_optimization/export_pt_bundle/sparse_best.pt
  params: 1,719,354
  flops : 0.64 GFLOPs

Ghost Pruned+Finetune:
  path  : /content/yolo_optimization/ghost_prune30_widthscale_320/weights/best.pt
  params: 1,042,131
  flops : 0.43 GFLOPs

=== Reduction vs Non-Ghost Baseline ===

Ghost Baseline:
  Params reduction: 1,291,884 (42.90%)
  FLOPs  reduction: 0.38 (37.24%)

Ghost Sparse:
  Params reduction: 1,291,884 (42.90%)
  FLOPs  reduction: 0.38 (37.24%)

Ghost Pruned+Finetune:
  Params reduction: 1,969,107 (65.39%)
  FLOPs  reduction: 0.60 (58.20%)



In [34]:
import os

def mb(p):
    return os.path.getsize(p) / (1024*1024)

for p in [
    "/content/yolo_optimization/ghost_baseline_320/weights/best.pt",
    "/content/yolo_optimization/export_pt_bundle/sparse_best.pt",
    "/content/yolo_optimization/ghost_prune30_widthscale_320/weights/best.pt",
]:
    print(p, f"{mb(p):.2f} MB")


/content/yolo_optimization/ghost_baseline_320/weights/best.pt 3.55 MB
/content/yolo_optimization/export_pt_bundle/sparse_best.pt 3.54 MB
/content/yolo_optimization/ghost_prune30_widthscale_320/weights/best.pt 2.26 MB


In [35]:
import torch
from ultralytics import YOLO

def sparsity_ratio(pt, eps=1e-3):
    m = YOLO(pt).model
    total = 0
    zeroish = 0
    with torch.no_grad():
        for p in m.parameters():
            if p.ndim == 0:
                continue
            t = p.detach().abs()
            total += t.numel()
            zeroish += (t < eps).sum().item()
    return zeroish / total if total else 0.0

pts = {
    "Ghost Baseline": "/content/yolo_optimization/ghost_baseline_320/weights/best.pt",
    "Ghost Sparse": "/content/yolo_optimization/export_pt_bundle/sparse_best.pt",
    "Ghost Pruned+FT": "/content/yolo_optimization/ghost_prune30_widthscale_320/weights/best.pt",
}

for k,v in pts.items():
    print(k, f"sparsity(<1e-3): {sparsity_ratio(v):.3f}")


Ghost Baseline sparsity(<1e-3): 0.031
Ghost Sparse sparsity(<1e-3): 0.030
Ghost Pruned+FT sparsity(<1e-3): 0.024


In [20]:
from ultralytics.nn.modules.conv import GhostConv
GhostConv??


In [21]:
from ultralytics.nn.modules.block import GhostBottleneck
GhostBottleneck??
